In [13]:
# import necessary libraries
import pandas as pd
import numpy as np
import statsmodels.api as sm
from sklearn.linear_model import LogisticRegression 
from scipy.spatial.distance import mahalanobis


## Question 1

Using homework_8.1.csv, find the Average treatment effect with inverse probability weighting. Then, include your code and a written explanation of your work (mentioning any choices or strategies you made in writing the code) in your homework reflection.    

Here are some steps to follow:    

* Estimate the propensity scores using logistic regression. Fit the model so that the Z values predict ﻿X﻿. 

* Use the model to predict the propensity scores (e.g., using predict_proba if you are using sklearn). 

* Calculate inverse probability weights (1/P for X=1 and 1/1-p for X=0). 

* Estimate the average treatment effect (the Y difference between X=1 and X=0, using the appropriate weights for each).    

Then, the ATE is closest to: 

- A: 2.0

- B: 2.3

- C: 2.6

- D: 2.9

In [2]:
df = pd.read_csv('../data/homework_8.1.csv')
df.head()

,X,Y,Z
0,1,4.109218,1.764052
1,0,2.259504,0.400157
2,0,-0.647584,0.978738
3,0,2.106071,2.240893
4,1,3.583464,1.867558


In [ ]:
# way 1: using statsmodels for logistic regression
X = sm.add_constant(df["Z"])
y = df["X"]
model = sm.Logit(y, X).fit()
y_pred = model.predict(X)
print(y_pred[:10])

Optimization terminated successfully.
         Current function value: 0.604614
         Iterations 5
0    0.841702
1    0.585325
2    0.712446
3    0.894224
4    0.854656
5    0.269971
6    0.706705
7    0.452238
8    0.463858
9    0.587787
dtype: float64


In [ ]:
# way 2: using sklearn for logistic regression
X_sklearn = df[["Z"]]
model_sklearn = LogisticRegression()
model_sklearn.fit(X_sklearn, y)
y_pred_sklearn = model_sklearn.predict_proba(X_sklearn)[:, 1]
print(y_pred_sklearn[:10])

[0.84011371 0.58464597 0.71108245 0.89279265 0.85308892 0.27122817
 0.70536505 0.45245487 0.46399577 0.58709256]


In [ ]:
# calculate the inverse probability weights (IPW)
df["IPW"] = np.where(df["X"] == 1, 1 / y_pred, 1 / (1 - y_pred))

# calculate the weighted means for treated and control groups
treated = df.loc[df["X"] == 1, ["Y", "IPW"]]
treat_weighted_mean = np.sum(treated["Y"] * treated["IPW"]) / np.sum(treated["IPW"])
# treat_weighted_mean = np.average(treated["Y"],weights=treated["IPW"])

control = df.loc[df["X"] == 0, ["Y", "IPW"]]
control_weighted_mean = np.sum(control["Y"] * control["IPW"]) / np.sum(control["IPW"])
# control_weighted_mean = np.average(control["Y"], weights=control["IPW"])
ate_ipw = treat_weighted_mean - control_weighted_mean
ate_ipw

np.float64(2.2680432299497104)

Answer : B  
_____

## Question 2
The propensity scores of the first three items in the dataframe are closest to: 

- A: [0.84, 0.58, 0.71] 

- B: [0.28, 0.34, 0.92]

- C: [0.08, 0.65, 0.38]

- D: [0.74, 0.39, 0.52]

In [11]:
y_pred[:3]

0    0.841702
1    0.585325
2    0.712446
dtype: float64

Answer :  A
_____

## Question 3
Using homework_8.2.csv, match all treated items to the single nearest untreated item using the Mahalanobis distance. (Do this with replacement — the same untreated item can be used again.)    


* Use the Mahalanobis function from scipy.spatial.distance 

* For the inverse covariance matrix, use all Z1 values and all Z2 values, make them into a 2xN matrix, find its 2x2 covariance, and invert.   


Then, the ATE is closest to: 

- A: 2.5

- B: 2.8

- C: 3.1

- D: 3.4

In [12]:
df2 = pd.read_csv('../data/homework_8.2.csv')
df2.head()

,X,Y,Z1,Z2
0,1,4.652085,1.764052,2.320015
1,1,2.590221,0.400157,1.292631
2,1,3.898981,0.978738,0.556423
3,1,5.857179,2.240893,2.345607
4,1,3.647489,1.867558,2.095611


In [ ]:
# # 1. Separate treated and control groups
# treated = df2[df2["X"] == 1]
# control = df2[df2["X"] == 0]

# # 2. Calculate inverse covariance matrix
# Z = df2[["Z1", "Z2"]].to_numpy().T
# S = np.cov(Z)
# VI = np.linalg.inv(S)

# # 3. Match each treated item with its nearest control
# matches = []

# for t_idx, t in treated.iterrows():

#     u = t[["Z1", "Z2"]].to_numpy(dtype=float)

#     distances = []

#     for c_idx, c in control.iterrows():

#         v = c[["Z1", "Z2"]].to_numpy(dtype=float)

#         d = mahalanobis(u, v, VI)

#         distances.append(d)

#     best_pos = np.argmin(distances)

#     best_control = control.iloc[best_pos]
#     best_idx = control.index[best_pos]

#     matches.append({
#         "treated_id": t_idx,
#         "control_id": best_idx,
#         "distance": distances[best_pos],
#         "Y_treated": t["Y"],
#         "Y_control": best_control["Y"],
#         "effect": t["Y"] - best_control["Y"]
#     })

# # 4. Create matching results DataFrame
# matched_df = pd.DataFrame(matches)

# # 5. Calculate average treatment effect for treated items
# ate = matched_df["effect"].mean()

# print("Estimated treatment effect:", ate)
# print(matched_df.head(10))

In [14]:
treated = df2[df2["X"] == 1]
control = df2[df2["X"] == 0]

Z_t = treated[["Z1", "Z2"]].to_numpy()
Z_c = control[["Z1", "Z2"]].to_numpy()

VI = np.linalg.inv(np.cov(df2[["Z1", "Z2"]].to_numpy().T))

matches = [np.argmin([mahalanobis(t, c, VI) for c in Z_c]) for t in Z_t]

ate = np.mean(treated["Y"].to_numpy() - control["Y"].to_numpy()[matches])

print("Estimated effect:", ate)

Estimated effect: 3.437678997911326


Answer :  D
_____

## Question 4
Find the nearest Z1 and Z2 values of the treated item with the least common support (the farthest Mahalanobis distance from the untreated). 

- A: (2.3, 1.2) 

- B: (0.2, -0.4) 

- C: (0.9, 1.4) 

- D: (1.5, -1.3) 

In [ ]:
# way 1: using the nearest distance for each treated item
min_distance = [np.min([mahalanobis(t, c, VI) for c in Z_c]) for t in Z_t]
idx_max = np.argmax(min_distance)
treated.iloc[idx_max][["Z1", "Z2"]]


Z1    2.696224
Z2    0.538155
Name: 494, dtype: float64

In [38]:
# way 2: using the nearest distance for each treated item
nearest_distances = [mahalanobis(t, Z_c[m], VI)for t, m in zip(Z_t, matches)]

# Find treated item with the largest nearest distance
worst_idx = np.argmax(nearest_distances)

# Show its Z1 and Z2
print(f" treated item with the largest nearest distance: {worst_idx}")
print(f" Z1: {treated.iloc[worst_idx]['Z1']}, Z2: {treated.iloc[worst_idx]['Z2']}")
print("Distance:", nearest_distances[worst_idx])

# Its nearest control
print()
print(control.iloc[matches[worst_idx]][["Z1", "Z2"]])

 treated item with the largest nearest distance: 241
 Z1: 2.69622405256358, Z2: 0.5381554886023228
Distance: 1.3830045330353775

Z1    1.519995
Z2   -1.282208
Name: 418, dtype: float64


Answer :  D
_____